# Chapter 20: Observability & Guardrails
## Topic 4: Output Validation + Schema Enforcement

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This exists

- Topic 3 addressed the risk of malicious or manipulated *input* reaching this project's pipeline. This topic addresses the complementary, equally necessary concern at the *other* end of the pipeline: even with a well-defended input path, a language model's *output* is itself a probabilistic, imperfect generation — it can produce a malformed classification label, an incorrectly-structured tool-call request, or a response that simply doesn't conform to what the rest of this project's pipeline expects to receive, entirely independent of any injection or malicious input.
- **Output validation and schema enforcement**, precisely: explicitly checking that a model's generated output actually conforms to the specific, expected structure before that output is used for anything consequential — a classification result should be exactly one of the three valid labels (FD, Non-FD, Multiple Category), not some near-miss variant; a tool-call request should match Chapter 10 Topic 4's own real, documented schema exactly, not an almost-correct approximation the receiving code might silently mishandle.
- Why this is a genuinely necessary, distinct guardrail, not redundant with this notebook series' existing evaluation work: Chapter 15's evaluation framework checks whether the model's output was *correct* (the right label, the right tool call) — this topic's validation checks something more basic and more immediate: whether the output is even *well-formed* enough to process at all, a check that needs to happen at actual runtime, on every single real request, not just during a periodic evaluation run — an output can be perfectly well-formed and still wrong (Chapter 15's concern), or it can be malformed in a way that would crash or silently corrupt downstream processing entirely, regardless of whether its intended content was even correct.


### 2. Internal Working — Step by Step

**Designing and applying genuine output validation to this project's actual pipeline, step by step:**

1. **Define the exact, explicit expected schema for each kind of model output this project's pipeline produces** — the classification output must be exactly one of three specific string values; a tool-call request must match Chapter 10 Topic 4's own real, already-documented tool schemas precisely (correct argument names, correct types); a generated customer-facing response should meet basic structural expectations (non-empty, within a reasonable length range).
2. **Validate every real output against its defined schema immediately upon generation, before it's used for anything downstream** — this validation step happens at actual runtime, on every single request, not as a periodic, after-the-fact evaluation check (Chapter 15's separate, complementary concern).
3. **Define an explicit, deliberate handling policy for validation failures** — a classification output that doesn't match any of the three valid labels, or a tool-call request with a malformed argument, needs a specific, decided response: reject and retry the generation call, fall back to a safe default (Topic 5's specific subject), or escalate for human review — leaving this undefined risks a validation failure silently corrupting downstream processing in an unplanned, unpredictable way.
4. **Log every validation failure with genuinely useful detail** (exactly what was expected, exactly what was actually received) as its own span within Topic 1's completed tracing infrastructure — this is what turns validation from a purely defensive, silent check into genuine, ongoing operational intelligence about how often and in what specific ways this project's actual model outputs deviate from expected structure.


### 3. How This Is Implemented in This Project

- This project's actual classification task has an exact, small, enumerable set of valid outputs (FD, Non-FD, Multiple Category, established since Chapter 1) — making classification-output validation a simple, high-confidence, unambiguous check: does the generated output exactly match one of these three specific strings, with no tolerance for near-miss variants a downstream system might silently mishandle.
- Chapter 10 Topic 4's own real, already-documented tool schemas (for `validate_fd_reference`, `search_knowledge_base`, `lookup_product_catalog`, `check_sender_history`) are directly reusable here as the exact, explicit validation target for tool-call outputs — this topic's validation mechanism doesn't need to invent new schema definitions, but should validate every real tool-call request against these already-established, real specifications before actually executing the call.
- This project's regulated, NBFC context makes validation failure handling (Topic 4's own explicit policy question) a genuinely consequential design decision, not a minor implementation detail — given Chapter 15 Topic 3's own real risk-prioritization (identifying `validate_fd_reference` as this project's highest-risk tool), a malformed tool-call request targeting that specific tool should trigger the most conservative failure-handling response (reject and escalate, not silently retry or approximate), directly connecting this topic's validation policy to this project's own already-established risk understanding.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **A validation check that's too permissive (accepting near-miss or loosely-matching output) provides little genuine protection**, defeating the purpose of having explicit schema enforcement at all — validation needs to be genuinely strict against the actual, defined schema, not a loose, best-effort approximation that would let most malformed output through anyway.
- **A validation check that's too strict, rejecting genuinely acceptable output due to an overly narrow schema definition, creates real, unnecessary failures** — if the schema itself doesn't correctly capture every genuinely valid output variant (a correctly-formatted but slightly differently-cased label, for instance), overly strict validation would incorrectly reject good output, requiring the schema definition itself to be carefully, deliberately calibrated against real, observed model behavior.
- **Validation failure handling needs to distinguish severity based on what specifically failed and for which specific downstream use**, exactly mirroring Chapter 15 Topic 3's own risk-based prioritization — a malformed classification label is a real but comparatively lower-severity failure (perhaps handled with a safe default or retry); a malformed tool-call request targeting `validate_fd_reference` specifically is a higher-severity failure warranting more conservative handling, given that tool's demonstrated real risk.
- **Debugging a recurring validation failure requires investigating the actual, specific pattern of malformed output being produced** — using Topic 1's completed tracing infrastructure to inspect exactly what the model generated in failing cases, checking whether this reflects a genuine, systematic model-behavior issue (perhaps needing a prompt refinement, Chapter 10 Topic 4's own evidence-based validation discipline) or a one-off, rare probabilistic anomaly not worth over-engineering a fix for.
- **Monitoring:** tracking validation failure rates over time, broken down by output type (classification vs tool-call vs generated response) and specifically by which tool a failed tool-call request targeted, reveals whether this project's actual model behavior remains well-calibrated against its defined schemas, or whether a prompt or model change has introduced a new, systematic validation-failure pattern requiring investigation.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Strict, exact-match schema validation vs a more lenient, fuzzy-matching approach:** strict validation (this topic's recommended default) provides genuine, reliable protection against malformed output reaching downstream processing, at the cost of potentially rejecting some genuinely acceptable output if the schema definition itself isn't perfectly calibrated — the right approach is strict validation against a carefully, deliberately calibrated schema, not loosening validation itself to compensate for an imprecise schema definition.
- **Reject-and-retry vs fall back to a safe default vs escalate for human review, as the validation-failure handling policy:** this should vary by the specific output type and its downstream consequence — a malformed classification output might reasonably trigger a retry (given classification's comparatively lower individual-request stakes); a malformed tool-call request targeting this project's highest-risk tool should more conservatively escalate rather than retry or approximate, directly connecting this policy decision to Chapter 15 Topic 3's own risk-based framework.
- **How much validation-failure detail to log:** enough to genuinely support debugging and pattern investigation (the exact expected schema, the exact actual output received) without excessive verbosity, mirroring Chapter 14 Topic 4's own "log enough detail to be useful" principle now applied specifically to validation-failure logging.


### 6. Alternatives and When to Use Each

- **No explicit output validation (relying solely on the model reliably producing well-formed output, this project's actual situation prior to this topic):** risks silent downstream corruption or failure whenever the model's probabilistic generation deviates from expected structure, a real, unaddressed risk this topic's analysis identifies.
- **Strict, explicit schema validation at runtime, on every real request (this topic's actual, recommended approach):** the right, necessary safeguard, directly complementing Chapter 15's separate, periodic evaluation-based correctness checking rather than duplicating it.
- **A more lenient, best-effort validation approach:** not recommended as a substitute for strict validation — if leniency is genuinely needed for specific, known-acceptable output variants, this should be reflected in a carefully expanded, deliberate schema definition, not in loosening the validation check itself.


### 7. Common Mistakes and Production Failures

- Not validating model output against an explicit schema at all, allowing occasional malformed output to silently corrupt or crash downstream processing without any warning or graceful handling.
- Validating too permissively, accepting near-miss or loosely-matching output that provides little genuine protection against the actual risk this guardrail exists to address.
- Validating too strictly against an imprecisely-calibrated schema, incorrectly rejecting genuinely acceptable output variants the schema definition failed to properly account for.
- Applying the same, undifferentiated failure-handling policy to every kind of validation failure, rather than calibrating response severity to the specific output type and its actual downstream risk (mirroring Chapter 15 Topic 3's risk-prioritization framework).
- Not logging validation failures with genuinely useful detail, missing the opportunity to identify and investigate systematic patterns in how this project's actual model output deviates from expected structure over time.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What does output validation check, and how is this different from Chapter 15's evaluation framework?
  A: Output validation checks whether a model's generated output is well-formed — conforms to its expected schema — at actual runtime, on every single real request. Chapter 15's evaluation checks whether output is correct (the right label, the right tool call), typically during periodic evaluation runs. An output can be well-formed but wrong (Chapter 15's concern), or malformed in a way that would corrupt downstream processing entirely regardless of intended correctness (this topic's concern) — these are genuinely different, complementary checks.

- Q: Why is this project's classification task a simple, high-confidence validation target?
  A: The classification task has an exact, small, enumerable set of valid outputs (FD, Non-FD, Multiple Category) — validation simply needs to confirm the generated output exactly matches one of these three specific strings, with no tolerance for near-miss variants, making this an unambiguous, straightforward check.

**Intermediate**

- Q: Why should validation-failure handling policy vary based on which specific tool a malformed tool-call request targeted?
  A: This directly connects to Chapter 15 Topic 3's own real risk-prioritization — a malformed request targeting `validate_fd_reference` (this project's demonstrated highest-risk tool, given its cross-customer-data-leak potential) warrants a more conservative response (escalation, not retry or approximation) than a malformed request targeting a lower-stakes tool, since the consequence of proceeding incorrectly differs substantially by tool.

- Q: Why is validating too permissively as much a real problem as validating too strictly?
  A: Overly permissive validation (accepting near-miss or loosely-matching output) provides little genuine protection, defeating the purpose of having explicit schema enforcement at all — malformed output that should have been caught could still reach downstream processing, silently reproducing the exact risk this guardrail was meant to eliminate.

**Advanced**

- Q: Design a complete output-validation strategy for this project, covering classification output, tool-call requests, and generated responses.
  A: For classification output, validate an exact match against the three defined labels, with any non-matching output rejected and retried (a lower-stakes failure category). For tool-call requests, validate exactly against Chapter 10 Topic 4's own documented schemas, with failure-handling severity calibrated by which specific tool was targeted — `validate_fd_reference` failures escalate to human review given that tool's demonstrated highest risk, while lower-risk tools might reasonably retry. For generated customer-facing responses, validate basic structural expectations (non-empty, within a reasonable length range, absence of any leaked system-prompt content, connecting to Topic 3's own injection-related concern). Log every validation failure, with genuinely useful detail, as its own span within Topic 1's completed tracing infrastructure, enabling ongoing pattern investigation.

- Q: A teammate proposes skipping output validation for tool-call requests specifically, arguing that Chapter 10 Topic 5's existing error-handling (retryable vs non-retryable errors) already provides sufficient protection. What's your concern?
  A: Chapter 10 Topic 5's error handling addresses failures during a tool's actual *execution* (a timeout, a downstream service error) — it doesn't address the genuinely earlier, distinct concern of whether the tool-call *request itself* was even well-formed before execution was attempted at all. A malformed request (an incorrect argument type or a missing required field) could either fail unpredictably during execution in a way Chapter 10 Topic 5's error handling wasn't designed to anticipate, or worse, execute "successfully" with subtly incorrect arguments — this topic's validation specifically catches malformation before execution, a genuinely earlier and distinct safeguard from execution-time error handling.

**Scenario-based**

- Q: Output-validation monitoring reveals a sudden, sharp increase in malformed tool-call requests specifically targeting `validate_fd_reference`, following a recent system-prompt update. Walk through your response.
  A: Given this tool's demonstrated highest-risk status (Chapter 15 Topic 3), this warrants immediate investigation rather than routine handling — use Topic 1's completed tracing infrastructure to inspect the specific pattern of malformed requests being generated, checking whether the recent prompt update inadvertently introduced ambiguity or confusion about this tool's expected argument structure. This points toward either reverting the recent prompt change or specifically refining it to restore clear, correct tool-call formatting, followed by re-validating (mirroring Chapter 10 Topic 4's evidence-based validation discipline) that the fix genuinely resolves the malformed-request pattern before considering the issue closed.

**Follow-up questions to expect:**

- "How would you decide the right balance between strict validation and avoiding excessive false-positive rejections of genuinely acceptable output?"
- "What would you do if a specific validation failure pattern couldn't be resolved through prompt refinement alone?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Output validation against an explicit schema is a specific application of a much more general software-engineering principle: defensive programming, and specifically the "validate at the boundary" pattern** — any system receiving data from an external or unpredictable source (here, a probabilistic model's generation) should validate that data at the point it enters the system, rather than assuming it's well-formed and discovering otherwise only when downstream processing fails unpredictably.
- **The distinction between "well-formed" and "correct" this topic establishes is a specific instance of a general distinction in formal verification and type theory**: syntactic validity (does the output have the right shape/type) versus semantic correctness (does the output have the right, intended meaning/value) — two genuinely different properties requiring different verification mechanisms, exactly as this topic's validation and Chapter 15's evaluation address different, complementary concerns.
- **This topic's validation mechanism is the direct, necessary input to Topic 5's fallback design**: a validation failure is precisely one of the specific trigger conditions Topic 5's fallback logic needs to handle gracefully, making these two topics directly, sequentially connected rather than independent concerns.

### 10. Quick Revision Summary (for last-minute interview prep)

> Output validation checks whether a model's generated output is well-formed — conforms to its expected schema — at actual runtime, on every real request, a genuinely different and complementary concern from Chapter 15's evaluation framework, which checks whether output is correct. This project's classification task has a simple, exact, three-value schema (FD, Non-FD, Multiple Category) making validation unambiguous; tool-call requests should be validated exactly against Chapter 10 Topic 4's own already-documented schemas before execution is even attempted. Validation-failure handling policy should vary by output type and its actual downstream risk, directly reusing Chapter 15 Topic 3's own risk-prioritization framework — a malformed request targeting this project's highest-risk tool (`validate_fd_reference`) warrants conservative escalation, not retry or silent approximation. Every validation failure should be logged with genuinely useful detail as its own span within Topic 1's completed tracing infrastructure, turning validation from a purely defensive, silent check into ongoing operational intelligence about how this project's real model behavior deviates from expected structure over time — directly setting up Topic 5's fallback design, since a validation failure is precisely one of the specific conditions that logic needs to handle gracefully.


### Module 1: Real Schema Validation — Classification Output and Tool-Call Requests

In [1]:

# ------------------------------------------------------------------
# MODULE 1: REAL schema validation, tested against real and malformed outputs
# ------------------------------------------------------------------

VALID_LABELS = {"FD", "Non-FD", "Multiple Category"}

# Chapter 10 Topic 4's REAL, documented tool schemas, reused here
TOOL_SCHEMAS = {
    "validate_fd_reference": {"reference_number": str},
    "search_knowledge_base": {"query": str},
    "lookup_product_catalog": {"product_name": str},
    "check_sender_history": {"sender_email": str},
}

def validate_classification_output(output):
    is_valid = output in VALID_LABELS
    return {"valid": is_valid, "expected": "one of FD/Non-FD/Multiple Category", "received": output}

def validate_tool_call(tool_name, arguments):
    if tool_name not in TOOL_SCHEMAS:
        return {"valid": False, "reason": f"unknown tool '{tool_name}'"}
    expected_schema = TOOL_SCHEMAS[tool_name]
    for arg_name, expected_type in expected_schema.items():
        if arg_name not in arguments:
            return {"valid": False, "reason": f"missing required argument '{arg_name}'"}
        if not isinstance(arguments[arg_name], expected_type):
            return {"valid": False, "reason": f"argument '{arg_name}' has wrong type"}
    return {"valid": True, "reason": None}


print("=" * 70)
print("MODULE 1: REAL SCHEMA VALIDATION -- CLASSIFICATION AND TOOL CALLS")
print("=" * 70)

classification_test_cases = ["FD", "Non-FD", "Multiple Category", "fd", "Fixed Deposit", ""]
print("\nClassification output validation:")
for output in classification_test_cases:
    result = validate_classification_output(output)
    print(f"  '{output}' -> valid={result['valid']}")

tool_call_test_cases = [
    ("validate_fd_reference", {"reference_number": "BJ2019FD7717"}),  # WELL-FORMED
    ("validate_fd_reference", {}),                                     # MISSING argument
    ("validate_fd_reference", {"reference_number": 12345}),            # WRONG type
    ("search_knowledge_base", {"query": "penalty policy"}),            # WELL-FORMED
]
print("\nTool-call request validation:")
for tool_name, args in tool_call_test_cases:
    result = validate_tool_call(tool_name, args)
    print(f"  {tool_name}({args}) -> valid={result['valid']}, reason={result['reason']}")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL SCHEMA VALIDATION -- CLASSIFICATION AND TOOL CALLS

Classification output validation:
  'FD' -> valid=True
  'Non-FD' -> valid=True
  'Multiple Category' -> valid=True
  'fd' -> valid=False
  'Fixed Deposit' -> valid=False
  '' -> valid=False

Tool-call request validation:
  validate_fd_reference({'reference_number': 'BJ2019FD7717'}) -> valid=True, reason=None
  validate_fd_reference({}) -> valid=False, reason=missing required argument 'reference_number'
  validate_fd_reference({'reference_number': 12345}) -> valid=False, reason=argument 'reference_number' has wrong type
  search_knowledge_base({'query': 'penalty policy'}) -> valid=True, reason=None

Module 1 complete. Run Module 2 next.


### Module 2: Risk-Calibrated Failure Handling — Reusing Chapter 15 Topic 3's Risk Profile

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Risk-calibrated failure handling, reusing Ch15 Topic 3
# ------------------------------------------------------------------

# Chapter 15 Topic 3's REAL, computed risk profile, reused directly
TOOL_RISK_PROFILE = {
    "validate_fd_reference": {"risk_score": 9, "consequence": "cross-customer data leak"},
    "lookup_product_catalog": {"risk_score": 6, "consequence": "wrong product figures stated"},
    "search_knowledge_base": {"risk_score": 5, "consequence": "generic/incomplete answer"},
    "check_sender_history": {"risk_score": 3, "consequence": "missed/unnecessary context"},
}

def determine_failure_handling_policy(tool_name, risk_profile):
    risk_score = risk_profile.get(tool_name, {}).get("risk_score", 5)
    if risk_score >= 8:
        return "ESCALATE_TO_HUMAN_REVIEW"
    elif risk_score >= 5:
        return "REJECT_AND_RETRY"
    else:
        return "LOG_AND_USE_SAFE_DEFAULT"


print("=" * 70)
print("MODULE 2: RISK-CALIBRATED FAILURE HANDLING POLICY")
print("=" * 70)

malformed_cases = [
    ("validate_fd_reference", {}),           # missing arg -- HIGHEST risk tool
    ("lookup_product_catalog", {}),          # missing arg -- medium risk
    ("check_sender_history", {}),            # missing arg -- lowest risk
]

print(f"\n{'Tool':<25} | {'Risk Score':>10} | {'Failure Handling Policy':>28}")
print("-" * 70)
for tool_name, args in malformed_cases:
    validation = validate_tool_call(tool_name, args)
    policy = determine_failure_handling_policy(tool_name, TOOL_RISK_PROFILE)
    risk = TOOL_RISK_PROFILE[tool_name]["risk_score"]
    print(f"{tool_name:<25} | {risk:>10} | {policy:>28}")

print(f"\nCONFIRMED: a malformed request targeting validate_fd_reference (risk 9,")
print(f"the SAME tool Chapter 15 Topic 3 identified as this project's highest-risk)")
print(f"correctly triggers the MOST conservative policy (ESCALATE), while the")
print(f"SAME kind of malformation (missing argument) on check_sender_history")
print(f"(risk 3) correctly triggers a MUCH lighter response -- proving failure")
print(f"handling severity is CALIBRATED to actual, real risk, not applied")
print(f"uniformly regardless of which tool actually failed.")

print("\nModule 2 complete. All Chapter 20 Topic 4 modules done.")
print()
print("=" * 70)
print("CHAPTER 20 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("Real, executable schema validation for BOTH classification output")
print("(exact match against 3 valid labels) and tool-call requests")
print("(matching Chapter 10 Topic 4's real, documented schemas exactly)")
print("-- tested against both well-formed and genuinely malformed cases.")
print()
print("Failure-handling policy DIRECTLY REUSES Chapter 15 Topic 3's real")
print("risk-prioritization data -- demonstrated concretely: the SAME kind")
print("of malformation produces GENUINELY DIFFERENT handling severity")
print("depending on which specific tool was targeted.")
print()
print("This validation and risk-calibrated handling is the DIRECT input")
print("Topic 5's fallback design needs -- a validation failure is exactly")
print("one of the specific conditions that logic must handle gracefully.")


MODULE 2: RISK-CALIBRATED FAILURE HANDLING POLICY

Tool                      | Risk Score |      Failure Handling Policy
----------------------------------------------------------------------
validate_fd_reference     |          9 |     ESCALATE_TO_HUMAN_REVIEW
lookup_product_catalog    |          6 |             REJECT_AND_RETRY
check_sender_history      |          3 |     LOG_AND_USE_SAFE_DEFAULT

CONFIRMED: a malformed request targeting validate_fd_reference (risk 9,
the SAME tool Chapter 15 Topic 3 identified as this project's highest-risk)
correctly triggers the MOST conservative policy (ESCALATE), while the
SAME kind of malformation (missing argument) on check_sender_history
(risk 3) correctly triggers a MUCH lighter response -- proving failure
handling severity is CALIBRATED to actual, real risk, not applied
uniformly regardless of which tool actually failed.

Module 2 complete. All Chapter 20 Topic 4 modules done.

CHAPTER 20 TOPIC 4 -- KEY POINTS TO REMEMBER
Real, executable s